# Day 5 - Grounding inference

- status: implemented_toy_not_executed_real_model
- stage: VLM_DAY_5
- paper_ids: qwen3_vl_2025, refcoco_2014
- dataset_ids: synthetic_toy, user_selected_nas_data
- seed: 42
- scope: educational implementation; real inference/training is opt-in

이 노트북은 다른 사용자 파일, 공유 환경, checkpoint를 자동으로 변경하지 않는다. 실제 데이터는
`/nas/datahub/min` 아래 사용자가 지정한 경로만 읽는다.

## 1. Learning question

Qwen3-VL의 language response를 검증 가능한 box로 바꾸려면 prompt, JSON schema, coordinate convention, overlay를 어떻게 고정해야 하는가?

## 2. Background theory

Qwen3-VL은 relative coordinate 기반 point/box grounding을 지원한다. 이 과정은 prompt에 `bbox_2d`, `xyxy`, `0..1000`, strict JSON을 명시한다. model output은 신뢰하지 않고 parser에서 type, 길이, 범위, x1<=x2를 검증한다.

## 3. Paper connection

visual grounding은 표현과 referent region을 연결한다. Qwen3-VL의 autoregressive box 생성은 Grounding DINO의 decoder tensor와 출력 방식이 다르므로 같은 후처리를 적용하지 않는다.

## 4. Input/output and shapes

입력은 image + object/referring expression이다. 출력 schema는 `[{label: str, bbox_2d: [x1,y1,x2,y2]}]`. relative box를 원본 image W/H 기준 pixel `xyxy`로 변환한다.

In [ ]:
from pathlib import Path
import sys
import numpy as np

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (current, *current.parents) if (p / "pyproject.toml").is_file()),
    Path("/nas/home/mhlee/vlm-foundation-7days"),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("project:", PROJECT_ROOT)
print("numpy:", np.__version__)

## 5. Minimal implementation

strict JSON을 parse하고 1920x1080 pixel 좌표로 변환한다.

In [ ]:
import json
from vlm_foundation.grounding import grounding_prompt, parse_grounding_json

prompt = grounding_prompt(["safety helmet", "forklift"])
raw_response = json.dumps([
    {"label": "safety helmet", "bbox_2d": [100, 120, 360, 420]}
])
boxes = parse_grounding_json(raw_response)
pixel_boxes = [box.to_pixel(1920, 1080) for box in boxes]
print(prompt)
print(pixel_boxes)

## 6. Visualization sanity check

pixel box가 화면 범위에 있고 대상 위에 놓이는지 overlay한다. JSON validity만으로 spatial correctness를 보장하지 않는다.

In [ ]:
for box in pixel_boxes:
    x1, y1, x2, y2 = box.bbox_2d
    assert 0 <= x1 <= x2 <= 1920 and 0 <= y1 <= y2 <= 1080
print("pixel-boundary check: PASS")

## 7. Experiment

동일 target에 category, attribute, relation prompt를 사용하고 top-1 IoU와 JSON validity를 비교한다.

In [ ]:
prompts = [
    "helmet",
    "yellow safety helmet",
    "the helmet worn by the worker on the left",
]
print("Evaluate each prompt on one fixed image/GT:", prompts)

## 8. Metrics

JSON validity, box count, top-1 accuracy@IoU 0.5, mAP, prompt consistency, empty response를 분리한다.

## 9. Interpretation

생성된 좌표는 언어 token이다. 좌표 문법을 잘 생성하는 능력과 대상 위치를 정확히 이해하는 능력은 구분해 평가해야 한다.

## 10. Failure cases

Markdown fence, 잘린 JSON, x/y swap, resized/original coordinate 혼동, extreme aspect ratio, duplicate box, 존재하지 않는 객체 hallucination을 기록한다.

## 11. Real-service implications

schema invalid는 reject/retry하고, low-confidence 또는 안전 관련 결과는 human review로 보낸다. 좌표 convention과 prompt version을 prediction에 함께 저장한다.

## 12. Review questions

1. strict JSON parser가 필요한 이유는?
2. relative box를 pixel로 바꾸는 식은?
3. JSON validity와 IoU가 다른 이유는?
4. original/resized image 좌표 혼동을 어떻게 탐지하는가?
5. Grounding DINO와 Qwen3-VL grounding의 출력 방식 차이는?